In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 3, Finished, Available, Finished, False)

In [2]:
# Remove this before running Data Factory Pipeline
start_date = date. today() - timedelta(7)

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 4, Finished, Available, Finished, False)

In [3]:
# Load the JSON data into a Spark DataFrame
df = spark.read.option("multiline", "true").json(f"Files/{start_date}_earthquake_data.json")

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 5, Finished, Available, Finished, False)

In [4]:
df

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 6, Finished, Available, Finished, False)

DataFrame[geometry: struct<coordinates:array<double>,type:string>, id: string, properties: struct<alert:string,cdi:double,code:string,detail:string,dmin:double,felt:bigint,gap:bigint,ids:string,mag:double,magType:string,mmi:double,net:string,nst:bigint,place:string,rms:double,sig:bigint,sources:string,status:string,time:bigint,title:string,tsunami:bigint,type:string,types:string,tz:string,updated:bigint,url:string>, type: string]

In [5]:
df.head()

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 7, Finished, Available, Finished, False)

Row(geometry=Row(coordinates=[74.6193, 41.3441, 10.0], type='Point'), id='us6000sqfj', properties=Row(alert=None, cdi=None, code='6000sqfj', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000sqfj&format=geojson', dmin=0.392, felt=None, gap=78, ids=',us6000sqfj,', mag=4.2, magType='mb', mmi=None, net='us', nst=20, place='29 km WNW of Baetovo, Kyrgyzstan', rms=0.92, sig=271, sources=',us,', status='reviewed', time=1776297461324, title='M 4.2 - 29 km WNW of Baetovo, Kyrgyzstan', tsunami=0, type='earthquake', types=',origin,phase-data,', tz=None, updated=1776300574040, url='https://earthquake.usgs.gov/earthquakes/eventpage/us6000sqfj'), type='Feature')

In [6]:
# Reshape earthquake data
df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )
)


StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 8, Finished, Available, Finished, False)

In [7]:
df

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 9, Finished, Available, Finished, False)

DataFrame[id: string, longitude: double, latitude: double, elevation: double, title: string, place_description: string, sig: bigint, mag: double, magType: string, time: bigint, updated: bigint]

In [8]:
df.head()

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 10, Finished, Available, Finished, False)

Row(id='us6000sqfj', longitude=74.6193, latitude=41.3441, elevation=10.0, title='M 4.2 - 29 km WNW of Baetovo, Kyrgyzstan', place_description='29 km WNW of Baetovo, Kyrgyzstan', sig=271, mag=4.2, magType='mb', time=1776297461324, updated=1776300574040)

In [9]:
# Validate data: Check for missing or null values
df = (
    df
    .withColumn('longitude', when(isnull(col('longitude')), 0).otherwise(col('longitude')))
    .withColumn('latitude', when(isnull(col('latitude')), 0).otherwise(col('latitude')))
    .withColumn('time', when(isnull(col('time')), 0).otherwise(col('time')))
)

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 11, Finished, Available, Finished, False)

In [10]:
# Convert 'time' and 'updated' to timestamp
df = (
    df
    .withColumn('time', (col('time') / 1000).cast(TimestampType()))
    .withColumn('updated', (col('updated') / 1000).cast(TimestampType()))
)

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 12, Finished, Available, Finished, False)

In [11]:
# Append to the silver table
df.write.mode('append').saveAsTable('earthquake_events_silver')

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 13, Finished, Available, Finished, True)

In [12]:
notebookutils.notebook.exit({"start_date": start_date})

StatementMeta(, 96e9fd25-f642-4295-bb09-e1f47c768761, 14, Finished, Available, Finished, True)

ExitValue: {'start_date': datetime.date(2026, 4, 10)}